In [0]:
from pyspark.sql.functions import from_json, schema_of_json, col

In [0]:
# Step 1: Create sample data containing your exact JSON text strings
raw_data = [
    (1, '{"name": "John Doe", "age": 35, "address": {"city": "Anytown", "state": "CA"}, "children": [{"name": "Owen", "age": 10}, {"name": "Eva", "age": 8}]}'),
    (2, '{"name": "Kristi Doe", "age": 40, "address": {"city": "Anytown", "state": "CA"}, "children": [{"name": "Steve", "age": 10}]}')
]

In [0]:

# Load this into a DataFrame where the text column is named "json_column"
df = spark.createDataFrame(raw_data, ["id", "json_column"])

In [0]:
# Step 2: Extract the pure text string from the very first row of "json_column"
sample_json_string = df.select("json_column").first()[0]

In [0]:
df.printSchema()

root
 |-- id: long (nullable = true)
 |-- json_column: string (nullable = true)



In [0]:
display(df)

id,json_column
1,"{""name"": ""John Doe"", ""age"": 35, ""address"": {""city"": ""Anytown"", ""state"": ""CA""}, ""children"": [{""name"": ""Owen"", ""age"": 10}, {""name"": ""Eva"", ""age"": 8}]}"
2,"{""name"": ""Kristi Doe"", ""age"": 40, ""address"": {""city"": ""Anytown"", ""state"": ""CA""}, ""children"": [{""name"": ""Steve"", ""age"": 10}]}"


In [0]:
print(sample_json_string)

{"name": "John Doe", "age": 35, "address": {"city": "Anytown", "state": "CA"}, "children": [{"name": "Owen", "age": 10}, {"name": "Eva", "age": 8}]}


In [0]:
df.select("id").first()[0]

1

In [0]:
# Step 3: Let Databricks read the sample string and generate the blueprint automatically
auto_schema = schema_of_json(sample_json_string)

In [0]:
# Step 4: Parse the entire text column into an organized STRUCT column
parsed_df = df.withColumn("clean_user_info", from_json(col("json_column"), auto_schema))

In [0]:
display(parsed_df)

id,json_column,clean_user_info
1,"{""name"": ""John Doe"", ""age"": 35, ""address"": {""city"": ""Anytown"", ""state"": ""CA""}, ""children"": [{""name"": ""Owen"", ""age"": 10}, {""name"": ""Eva"", ""age"": 8}]}","List(List(Anytown, CA), 35, List(List(10, Owen), List(8, Eva)), John Doe)"
2,"{""name"": ""Kristi Doe"", ""age"": 40, ""address"": {""city"": ""Anytown"", ""state"": ""CA""}, ""children"": [{""name"": ""Steve"", ""age"": 10}]}","List(List(Anytown, CA), 40, List(List(10, Steve)), Kristi Doe)"


In [0]:
# Step 5: Pull the nested data out into clean, individual table columns
final_df = parsed_df.select(
    col("id"),
    col("clean_user_info.name").alias("parent_name"),
    col("clean_user_info.age").alias("parent_age"),
    col("clean_user_info.address.city").alias("city"),
    col("clean_user_info.address.state").alias("state"),
    col("clean_user_info.children").alias("children_list")
)

In [0]:
# Display the clean table output
display(final_df)

id,parent_name,parent_age,city,state,children_list
1,John Doe,35,Anytown,CA,"List(List(10, Owen), List(8, Eva))"
2,Kristi Doe,40,Anytown,CA,"List(List(10, Steve))"


JSON file is large, you cannot paste the whole string into a text function.
Instead, you let Databricks read the data file safely directly from storage. There are two simple ways to handle this depending on your goal:

In [0]:
from pyspark.sql.functions import from_json, schema_of_json, col

# 1. Read your large JSON file as a standard text table
df = spark.read.text("/path/to/large_json_file.json").toDF("json_column")

# 2. Automatically grab a sample string from the first row of your file
sample_row = df.select("json_column").first()[0]

# 3. Let Databricks build the schema blueprint automatically from that sample row
auto_schema = schema_of_json(sample_row)

# 4. Instantly unwrap the whole large file using the auto-derived blueprint!
parsed_df = df.withColumn("clean_data", from_json(col("json_column"), auto_schema))

display(parsed_df)

Define the schema using a simple string variable instead of StructType!

In [0]:
from pyspark.sql.functions import from_json, col

# 1. Create your dummy data
data = [("1", '{"user": "Zebi", "age": 28}')]
df = spark.createDataFrame(data, ["id", "raw_log"])

# 2. Define the schema using a simple string variable instead of StructType!
json_schema_string = "user STRING, age INT"

# 3. Unpack the JSON string using your string variable
parsed_df = df.withColumn("clean_user_info", from_json(col("raw_log"), json_schema_string))

# 4. Pull out the individual fields with a dot (.)
final_df = parsed_df.select(
    col("id"),
    col("clean_user_info.user").alias("username"),
    col("clean_user_info.age").alias("user_age")
)

display(final_df)

id,username,user_age
1,Zebi,28


In Python, we use the from_json function alongside schema_of_json or a manually defined schema to unwrap the text

In [0]:
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# 1. Create dummy data with a JSON string column
data = [("1", '{"user": "Zebi", "age": 28}')]
df = spark.createDataFrame(data, ["id", "raw_log"])

# 2. Define the schema (the "map") for your JSON content
json_schema = StructType([
    StructField("user", StringType(), True),
    StructField("age", IntegerType(), True)
])

# 3. Unpack the JSON string into a STRUCT column
parsed_df = df.withColumn("clean_user_info", from_json(col("raw_log"), json_schema))

# 4. Pull out individual nested fields using a dot (.)
final_df = parsed_df.select(
    col("id"),
    col("clean_user_info.user").alias("username"),
    col("clean_user_info.age").alias("user_age")
)

display(final_df)

id,username,user_age
1,Zebi,28


In [0]:
# 1. Re-create the clean JSON data records
raw_data = [
    ('{"user": "Zebi", "age": 28}',),
    ('{"user": "Peter", "age": 35}',)
]

# 2. Build the DataFrame with the column name 'raw_log'
df = spark.createDataFrame(raw_data, ["raw_log"])

# 3. Register it back as 'my_table' for your SQL queries
df.createOrReplaceTempView("my_table")

print("Table 'my_table' is successfully recreated and ready for SQL!")


In [0]:
json_column=['{"name": "John Doe", "age": 35, "address": {"city": "Anytown", "state": "CA"}, "children": [{"name": "Owen", "age": 10}, {"name": "Eva", "age": 8}]}'
'{"name": "Kristi Doe", "age": 40, "address": {"city": "Anytown", "state": "CA"}, "children": [{"name": "Steve", "age": 10}]}'] 


In [0]:
# 2. Build the DataFrame with the column name 'raw_log'
df = spark.createDataFrame(json_column, ["json_log"])

In [0]:




# 3. Register it back as 'my_table' for your SQL queries
df.createOrReplaceTempView("my_table")

print("Table 'my_table' is successfully recreated and ready for SQL!")

Table 'my_table' is successfully recreated and ready for SQL!


In [0]:

%sql

SELECT 
  -- 1. Convert the raw text string into a variant type
  parse_json(json_log) AS variant_log
FROM my_table;


variant_log
"{""address"":{""city"":""Anytown"",""state"":""CA""},""age"":35,""children"":[{""age"":10,""name"":""Owen""},{""age"":8,""name"":""Eva""}],""name"":""John Doe""}"


In [0]:
%sql
SELECT 
  parse_json(json_log):address:city AS city,
  parse_json(json_log):age::int AS user_age
FROM my_table;


city,user_age
"""Anytown""",35


In [0]:
%sql
SELECT 
  -- 1. Define the map/schema of your JSON
  -- 2. Use from_json to unpack it
  from_json(json_log, 'address STRING, age INT') AS clean_user_info
FROM my_table;

clean_user_info
"List({""city"":""Anytown"",""state"":""CA""}, 35)"


In [0]:
%sql
SELECT 
  -- 1. Use from_json to convert the text into our clean STRUCT blueprint
  from_json(json_log, '
    name STRING, 
    age INT, 
    address STRUCT<city STRING, state STRING>, 
    children ARRAY<STRUCT<name STRING, age INT>>
  ') AS user_struct
FROM my_table;


user_struct
"List(John Doe, 35, List(Anytown, CA), List(List(Owen, 10), List(Eva, 8)))"


In [0]:
# This lets you treat your Python 'df' like a SQL table named 'my_table'
df.createOrReplaceTempView("my_table")

# Run the SQL syntax directly on your DataFrame
parsed_df = spark.sql("""
  SELECT 
    -- 1. Convert text to a STRUCT using the blueprint schema
    from_json(json_log, '
      name STRING, 
      age INT, 
      address STRUCT<city STRING, state STRING>, 
      children ARRAY<STRUCT<name STRING, age INT>>
    ') AS user_struct
  FROM my_table
""")

# Show the results
display(parsed_df)



user_struct
"List(John Doe, 35, List(Anytown, CA), List(List(Owen, 10), List(Eva, 8)))"


In [0]:
%sql
SELECT schema_of_json('{"name": "John Doe", "age": 35, "address": {"city": "Anytown", "state": "CA"}, "children": [{"name": "Owen", "age": 10}, {"name": "Eva", "age": 8}]}');


"schema_of_json('{""name"": ""John Doe"", ""age"": 35, ""address"": {""city"": ""Anytown"", ""state"": ""CA""}, ""children"": [{""name"": ""Owen"", ""age"": 10}, {""name"": ""Eva"", ""age"": 8}]}')"
"STRUCT, age: BIGINT, children: ARRAY>, name: STRING>"


json_column1 is a Python list containing two JSON strings. We use a for loop to take each string and make it a separate row in the Spark DataFrame.

In [0]:
json_column1=['{"name": "John Doe", "age": 35, "address": {"city": "Anytown", "state": "CA"}, "children": [{"name": "Owen", "age": 10}, {"name": "Eva", "age": 8}]}'
'{"name": "Kristi Doe", "age": 40, "address": {"city": "Anytown", "state": "CA"}, "children": [{"name": "Steve", "age": 10}]}'] 


In [0]:
df = spark.createDataFrame(
    [(j,) for j in json_column],
    ["json_data"]
)

df.show(truncate=False)

+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|json_data                                                                                                                                                                                                                                                                       |
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|{"name": "John Doe", "age": 35, "address": {"city": "Anytown", "state": "CA"}, "children": [{"name": "Owen", "age": 10}, {"name": "Eva", "age": 8}]}{"name": "Kristi Doe", "ag

In [0]:
data = [(j,) for j in json_column1]

df = spark.createDataFrame(data, ["raw_log"])

df.show(truncate=False)

+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|raw_log                                                                                                                                                                                                                                                                         |
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|{"name": "John Doe", "age": 35, "address": {"city": "Anytown", "state": "CA"}, "children": [{"name": "Owen", "age": 10}, {"name": "Eva", "age": 8}]}{"name": "Kristi Doe", "ag

In [0]:
df.show(truncate=False)
df.printSchema()

+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|raw_log                                                                                                                                                                                                                                                                         |
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|{"name": "John Doe", "age": 35, "address": {"city": "Anytown", "state": "CA"}, "children": [{"name": "Owen", "age": 10}, {"name": "Eva", "age": 8}]}{"name": "Kristi Doe", "ag

In [0]:
from pyspark.sql.functions import schema_of_json, lit

sample_row = df.select("raw_log").first()[0]

schema_df = df.select(
    schema_of_json(lit(sample_row)).alias("json_schema")
)

schema_df.show(truncate=False)

+-----------------------------------------------------------------------------------------------------------------------------------+
|json_schema                                                                                                                        |
+-----------------------------------------------------------------------------------------------------------------------------------+
|STRUCT<address: STRUCT<city: STRING, state: STRING>, age: BIGINT, children: ARRAY<STRUCT<age: BIGINT, name: STRING>>, name: STRING>|
+-----------------------------------------------------------------------------------------------------------------------------------+



Why json_column1[0]? It retrieves the first JSON string from your Python list. There is no [1] because your list contains strings directly, not tuples with an ID and a JSON string.

In [0]:
df.select(
    schema_of_json(lit(json_column1[0])).alias("schema")
).show(truncate=False)

+-----------------------------------------------------------------------------------------------------------------------------------+
|schema                                                                                                                             |
+-----------------------------------------------------------------------------------------------------------------------------------+
|STRUCT<address: STRUCT<city: STRING, state: STRING>, age: BIGINT, children: ARRAY<STRUCT<age: BIGINT, name: STRING>>, name: STRING>|
+-----------------------------------------------------------------------------------------------------------------------------------+



In [0]:
from pyspark.sql.functions import schema_of_json, lit

df.select(
    schema_of_json(lit(data[0][0])).alias("schema")
).show(truncate=False)

+-----------------------------------------------------------------------------------------------------------------------------------+
|schema                                                                                                                             |
+-----------------------------------------------------------------------------------------------------------------------------------+
|STRUCT<address: STRUCT<city: STRING, state: STRING>, age: BIGINT, children: ARRAY<STRUCT<age: BIGINT, name: STRING>>, name: STRING>|
+-----------------------------------------------------------------------------------------------------------------------------------+



Data structure

	

How to get the first JSON string

json_column1 (list of strings)

	

json_column1[0]

data (list of one-value tuples)

	

data[0][0]

raw_data (list of ID + JSON tuples)

	

raw_data[0][1]

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW customers AS
SELECT * FROM VALUES
(
  '{"name": "John Doe", "age": 35, "address": {"city": "Anytown", "state": "CA"}, "children": [{"name": "Owen", "age": 10}, {"name": "Eva", "age": 8}]}'
),
(
  '{"name": "Kristi Doe", "age": 40, "address": {"city": "Anytown", "state": "CA"}, "children": [{"name": "Steve", "age": 10}]}'
)
AS t(raw_log);

In [0]:
%sql
SELECT * FROM customers;

raw_log
"{""name"": ""John Doe"", ""age"": 35, ""address"": {""city"": ""Anytown"", ""state"": ""CA""}, ""children"": [{""name"": ""Owen"", ""age"": 10}, {""name"": ""Eva"", ""age"": 8}]}"
"{""name"": ""Kristi Doe"", ""age"": 40, ""address"": {""city"": ""Anytown"", ""state"": ""CA""}, ""children"": [{""name"": ""Steve"", ""age"": 10}]}"


infer schemaa


In [0]:
%sql
SELECT schema_of_json(
  '{"name": "John Doe", "age": 35, "address": {"city": "Anytown", "state": "CA"}, "children": [{"name": "Owen", "age": 10}, {"name": "Eva", "age": 8}]}'
) AS json_schema;

#Step 4: Parse both JSON records using from_json()

In [0]:

%sql
SELECT
  from_json(
    raw_log,
    'STRUCT<
      name: STRING,
      age: BIGINT,
      address: STRUCT<city: STRING, state: STRING>,
      children: ARRAY<STRUCT<name: STRING, age: BIGINT>>
    >'
  ) AS customer
FROM customers;

customer
"List(John Doe, 35, List(Anytown, CA), List(List(Owen, 10), List(Eva, 8)))"
"List(Kristi Doe, 40, List(Anytown, CA), List(List(Steve, 10)))"


In [0]:
%sql
SELECT
  customer.name,
  customer.age,
  customer.address.city AS city,
  customer.children
FROM (
  SELECT from_json(
    raw_log,
    'STRUCT<
      name: STRING,
      age: BIGINT,
      address: STRUCT<city: STRING, state: STRING>,
      children: ARRAY<STRUCT<name: STRING, age: BIGINT>>
    >'
  ) AS customer
  FROM customers
);

name,age,city,children
John Doe,35,Anytown,"List(List(Owen, 10), List(Eva, 8))"
Kristi Doe,40,Anytown,"List(List(Steve, 10))"


In [0]:
%sql
DECLARE OR REPLACE VARIABLE json_schema STRING;

SET VAR json_schema = schema_of_json(
  '{"name":"John Doe","age":35,
    "address":{"city":"Anytown","state":"CA"},
    "children":[{"name":"Owen","age":10}]}'
);

In [0]:
%sql
SELECT
  from_json(raw_log, session.json_schema) AS customer
FROM customers;

customer
"List(List(Anytown, CA), 35, List(List(10, Owen), List(8, Eva)), John Doe)"
"List(List(Anytown, CA), 40, List(List(10, Steve)), Kristi Doe)"


In [0]:
schema_result = spark.sql("""
SELECT schema_of_json(
  '{"name":"John Doe","age":35,
    "address":{"city":"Anytown","state":"CA"},
    "children":[{"name":"Owen","age":10}]}'
) AS json_schema
""")

json_schema = schema_result.first()["json_schema"]

print(json_schema)

STRUCT<address: STRUCT<city: STRING, state: STRING>, age: BIGINT, children: ARRAY<STRUCT<age: BIGINT, name: STRING>>, name: STRING>


In [0]:
%sql
CREATE OR REPLACE TABLE customers
USING DELTA
AS
SELECT *
FROM read_files(
  '/path/to/customers.json',
  format => 'json'
);

In [0]:
%sql
SELECT schema_of_json(
  '{"name":"John","age":35}'
);

"schema_of_json('{""name"":""John"",""age"":35}')"
STRUCT


In [0]:
%sql
select from_json( '{"name":"John","age":35}','STRUCT<age: BIGINT, name: STRING>')

"from_json('{""name"":""John"",""age"":35}','STRUCT')"
"List(35, John)"


In [0]:
%sql
select parse_json('{"name":"John","age":35}')as data;

data
"{""age"":35,""name"":""John""}"


In [0]:
%sql
select  parse_json('{"name":"John","age":35}'):name as name,
       parse_json('{"name":"John","age":35}'):age as age;

name,age
"""John""",35


In [0]:
%sql
select  parse_json('{"name":"John","age":35}'):name::STRING as name,
       parse_json('{"name":"John","age":35}'):age as age;

name,age
John,35


In [0]:
%sql
SELECT schema_of_json(value) 
FROM read_files('/Volumes/my_catalog/my_schema/my_volume/large_file.json', format => 'text') 
LIMIT 1;


In [0]:
%sql
SELECT 
  -- Paste the output from Step 1 right here as the second argument
  from_json(value, 'name STRING, age INT, address STRUCT<city STRING, state STRING>, children ARRAY<STRUCT<name STRING, age INT>>') AS user_struct
FROM read_files('/Volumes/my_catalog/my_schema/my_volume/large_file.json', format => 'text');


In [0]:
%python
json_data = [
    '{"order_id":101,"amount":250}',
    '{"order_id":102,"amount":400}',
    '{"order_id":103,"amount":150}'
]

df = spark.createDataFrame(
    [(x,) for x in json_data],
    ["raw_json"]
)

display(df)

raw_json
"{""order_id"":101,""amount"":250}"
"{""order_id"":102,""amount"":400}"
"{""order_id"":103,""amount"":150}"


In [0]:
%python
from pyspark.sql.functions import schema_of_json, lit

schema = df.select(schema_of_json(lit(json_data[0])).alias("schema")).first()["schema"]

print(schema)

STRUCT<amount: BIGINT, order_id: BIGINT>


In [0]:
sample_row = df.select("raw_json").first()[0]

print(sample_row)


{"order_id":101,"amount":250}


another method----

In [0]:
schema_df = df.select(schema_of_json(lit(sample_row)).alias("json_schema"))

schema_df.show(truncate=False)

+----------------------------------------+
|json_schema                             |
+----------------------------------------+
|STRUCT<amount: BIGINT, order_id: BIGINT>|
|STRUCT<amount: BIGINT, order_id: BIGINT>|
|STRUCT<amount: BIGINT, order_id: BIGINT>|
+----------------------------------------+



In [0]:
sample_row = df.select("raw_log").first()[0]

schema_df = df.select(
    schema_of_json(lit(sample_row)).alias("json_schema")
)

schema_df.show(truncate=False)

-----------------

In [0]:
%python
from pyspark.sql.functions import from_json, col

parsed_df = df.withColumn("data",from_json(col("raw_json"), schema)
)

display(parsed_df.select("data.*"))

amount,order_id
250,101
400,102
150,103


In [0]:

display(parsed_df.select(col("data.amount").alias("usramt"),col("data.order_id")))

usramt,order_id
250,101
400,102
150,103


In [0]:
%sql
SELECT parse_json('{"order_id":101,"amount":250}') AS data;

data
"{""amount"":250,""order_id"":101}"


In [0]:
%sql
SELECT parse_json('{"order_id":101,"amount":250}'):order_id AS order_id;

order_id
101


In [0]:
from pyspark.sql.functions import struct, to_json

stream_df = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 1)
    .load()
)

json_stream = stream_df.select(
    to_json(
        struct(
            col("value").alias("order_id"),
            (col("value") * 10).alias("amount")
        )
    ).alias("raw_json")
)

display(
    json_stream,
    checkpointLocation="/Volumes/retail/sales/streaming_checkpoints/json_stream_checkpoint",outputMode="append"
)

query = (
    json_stream.writeStream
    .format("console")
    .outputMode("append")
    .option("checkpointLocation", "/Volumes/retail/sales/streaming_checkpoints/json_stream_checkpoint")
    .start()
)

Checkpointing to /Volumes/retail/sales/streaming_checkpoints/json_stream_checkpoint


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7386967445817867>, line 19
      3 stream_df = (
      4     spark.readStream
      5     .format("rate")
      6     .option("rowsPerSecond", 1)
      7     .load()
      8 )
     10 json_stream = stream_df.select(
     11     to_json(
     12         struct(
   (...)     16     ).alias("raw_json")
     17 )
---> 19 display(
     20     json_stream,
     21     checkpointLocation="/Volumes/retail/sales/streaming_checkpoints/json_stream_checkpoint",outputMode="append"
     22 )
     24 query = (
     25     json_stream.writeStream
     26     .format("console")
   (...)     29     .start()
     30 )

File <command-7386967445817867>, line 10
      1 from pyspark.sql.functions import struct, to_json
      3 stream_df = (
      4     spark.readStream
      5     .format("rate")
      6     .option("rowsPerSecond", 1)
      7 

Method 1 — Console first, then parse and save

Streaming DataFrame

Console output

Displays JSON for testing; does not persist it as a table

Parse JSON → VARIANT

Write parsed data to Delta

Important: console output is not a persistent storage location. You must separately write the JSON to a table or another durable sink before you can query it later with SQL.

In [0]:
from pyspark.sql.functions import col, struct, to_json

stream_df = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 1)
    .load()
)

json_stream = stream_df.select(
    to_json(
        struct(
            col("value").alias("order_id"),
            (col("value") * 10).alias("amount")
        )
    ).alias("raw_json")
)

query = (
    json_stream.writeStream
    .format("console")
    .outputMode("append")
    .option(
        "checkpointLocation",
        "/Volumes/retail/sales/streaming_checkpoints/json_stream_checkpoint_v2"
    )
    .trigger(availableNow=True)
    .start()
)

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS retail.sales.streaming_checkpoints;

In [0]:
display(
    json_stream,
    checkpointLocation="/Volumes/retail/sales/streaming_checkpoints/json_stream_checkpoint"
)

Checkpointing to /Volumes/retail/sales/streaming_checkpoints/json_stream_checkpoint


In [0]:
query.stop()

In [0]:
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, LongType

sales_schema = StructType([
    StructField("order_id", LongType(), True),
    StructField("amount", LongType(), True)
])

parsed_stream = json_stream.select(
    from_json(col("raw_json"), sales_schema).alias("data")
).select("data.*")

display(
    parsed_stream,
    checkpointLocation="/Volumes/retail/sales/streaming_checkpoints/parsed_stream_checkpoint"
)

Checkpointing to /Volumes/retail/sales/streaming_checkpoints/parsed_stream_checkpoint


In [0]:
query = (
    parsed_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        "/Volumes/retail/sales/streaming_checkpoints/sales_delta_checkpoint"
    )
    .trigger(availableNow=True)
    .toTable("sales_stream_demo")
)

In [0]:
%sql
SELECT *
FROM sales_stream_demo;

order_id,amount
0,0
8,80
16,160
24,240
32,320
40,400
48,480
56,560
64,640
72,720


Method 2 — Save raw JSON directly to Delta, then parse

Streaming DataFrame

Delta table (raw_json)

Persists the incoming JSON strings

SQL parse_json() → VARIANT

Extract individual fields

In [0]:
from pyspark.sql.functions import col, struct, to_json

stream_df = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 1)
    .load()
)

json_stream1 = stream_df.select(
    to_json(
        struct(
            col("value").alias("order_id"),
            (col("value") * 10).alias("amount")
        )
    ).alias("raw_json")
)

query = (
    json_stream1.writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        "/Volumes/retail/sales/streaming_checkpoints/raw_json_checkpoint"
    )
    .trigger(availableNow=True)
    .toTable("retail.sales.raw_sales_json")
)

In [0]:
%sql
SELECT COUNT(*) AS total_rows
FROM retail.sales.raw_sales_json;

total_rows
170


In [0]:
%sql
SELECT * FROM retail.sales.raw_sales_json;

raw_json
"{""order_id"":3,""amount"":30}"
"{""order_id"":11,""amount"":110}"
"{""order_id"":19,""amount"":190}"
"{""order_id"":27,""amount"":270}"
"{""order_id"":35,""amount"":350}"
"{""order_id"":43,""amount"":430}"
"{""order_id"":51,""amount"":510}"
"{""order_id"":59,""amount"":590}"
"{""order_id"":67,""amount"":670}"
"{""order_id"":75,""amount"":750}"


In [0]:
%sql
SELECT
    parse_json(raw_json) AS data
FROM retail.sales.raw_sales_json;

data
"{""amount"":30,""order_id"":3}"
"{""amount"":110,""order_id"":11}"
"{""amount"":190,""order_id"":19}"
"{""amount"":270,""order_id"":27}"
"{""amount"":350,""order_id"":35}"
"{""amount"":430,""order_id"":43}"
"{""amount"":510,""order_id"":51}"
"{""amount"":590,""order_id"":59}"
"{""amount"":670,""order_id"":67}"
"{""amount"":750,""order_id"":75}"


In [0]:
%sql
SELECT
    parse_json(raw_json):order_id AS order_id,
    parse_json(raw_json):amount AS amount
FROM retail.sales.raw_sales_json;

order_id,amount
3,30
11,110
19,190
27,270
35,350
43,430
51,510
59,590
67,670
75,750


In [0]:
%sql
SELECT
    CAST(parse_json(raw_json):order_id AS BIGINT) AS order_id,
    CAST(parse_json(raw_json):amount AS BIGINT) AS amount
FROM retail.sales.raw_sales_json;

order_id,amount
3,30
11,110
19,190
27,270
35,350
43,430
51,510
59,590
67,670
75,750


Method 2 is simpler when we want to learn SQL parse_json() and VARIANT, because the raw JSON is persisted in Delta and we can query it repeatedly.

One final distinction: parse_json() in our SQL query converts JSON to VARIANT when the query runs. It does not change the original raw_json column in the Delta table. If we want to persist the parsed VARIANT values, we can create another table using a CREATE TABLE AS SELECT query.

In [0]:
%sql
CREATE OR REPLACE TABLE retail.sales.parsed_sales AS
SELECT
    parse_json(raw_json) AS data
FROM retail.sales.raw_sales_json;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT
    data:order_id AS order_id,
    data:amount AS amount
FROM retail.sales.parsed_sales;

order_id,amount
0,0
8,80
16,160
24,240
32,320
40,400
48,480
56,560
64,640
72,720


In [0]:
%sql
SELECT
    *
FROM retail.sales.parsed_sales;

data
"{""amount"":0,""order_id"":0}"
"{""amount"":80,""order_id"":8}"
"{""amount"":160,""order_id"":16}"
"{""amount"":240,""order_id"":24}"
"{""amount"":320,""order_id"":32}"
"{""amount"":400,""order_id"":40}"
"{""amount"":480,""order_id"":48}"
"{""amount"":560,""order_id"":56}"
"{""amount"":640,""order_id"":64}"
"{""amount"":720,""order_id"":72}"
